So this is where i'm gonna start working on the educational version. I'd probably want a background on both peft and bayesian techniques here

In [2]:
pip install datasets transformers peft evaluate torchmetrics scikit-learn

   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   --------------------------------------- 527.0/527.0 kB 17.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   ---------------------------------------  10.0/10.1 MB 56.4 MB/s eta 0:00:01
   ---------------------------------------- 10.1/10.1 MB 48.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/557.0 kB ? eta -:--:--
   --------------------------------------- 557.0/557.0 kB 18.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/983.4 kB ? eta -:--:--
   --------------------------------------- 983.4/983.4 kB 23.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/625.2 kB ? eta -:--:--
   --------------------------------------- 625.2/625.2 kB 11.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/27.6 MB ? eta -:--:--
   ----------------- ---------------------- 12.3/27.6 MB 64.1 MB/s eta 0:00:01
   ----------------

In [4]:
from datasets import load_dataset

#Download the Rotten Tomatoes dataset directly from hugging face
raw_dataset = load_dataset("rotten_tomatoes")

#shrink it
#The original training set has 8,500 reviews. We will shuffle them and grab exactly 1,000.
#We will also grab 200 for your testing/evaluation set.
#1000 doesn't seem like a lot, but for deep learning on an unimpressive cpu, this is already a lot.
small_train_dataset = raw_dataset["train"].shuffle(seed=42).select(range(1000))
small_test_dataset = raw_dataset["test"].shuffle(seed=42).select(range(200))

print("\n--- Dataset Ready! ---") #yay!
print(f"Training examples: {len(small_train_dataset)}")
print(f"Testing examples: {len(small_test_dataset)}\n")

#A look at the very first example to see what we are working with
print(f"Text: '{small_train_dataset[0]['text']}'")
print(f"Label: {small_train_dataset[0]['label']} (0 = Negative, 1 = Positive)")


--- Dataset Ready! ---
Training examples: 1000
Testing examples: 200

Text: '. . . plays like somebody spliced random moments of a chris rock routine into what is otherwise a cliche-riddled but self-serious spy thriller .'
Label: 0 (0 = Negative, 1 = Positive)


In [ ]:
import torch
from modelwrappers.wrapperbase import WrapperBase

class EducationalBayesianWrapper(WrapperBase):
    def __init__(self, model, peft_config, args, accelerator, adapter_name="default"):
        #Initialize the professor's base class (handles the optimizer, metrics, etc.)
        super().__init__(model, peft_config, args, accelerator, adapter_name)

    def forward_logits(self, batch, sample=False, n_samples=1):
        """
        The Educational Engine:
        Takes a batch of text, passes it through RoBERTa, and returns the logits.
        """
        #Extract the text tokens and attention masks from the trimmed review batch
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']

        #standard pass through the model without sampling (just one guess)
        if not sample or n_samples == 1:
            outputs = self.base_model(
                input_ids=input_ids, 
                attention_mask=attention_mask
            )
            #The professor's evaluation math expects a 3D tensor: [batch_size, n_samples, classes]
            #So we add a dimension in the middle with unsqueeze(1)
            return outputs.logits.unsqueeze(1)

        #Bayesian pass
        else:
            #force the model into training mode to activate the random dropout layers
            self.base_model.train() 
            
            stacked_logits = []
            
            #Run the exact same text through the model n_samples times (e.g., 10 times)
            for _ in range(n_samples):
                outputs = self.base_model(
                    input_ids=input_ids, 
                    attention_mask=attention_mask
                )
                stacked_logits.append(outputs.logits)
            
            #safely return the model to evaluation mode
            self.base_model.eval()

            #stack all 10 guesses together into a single block of math
            #Final Shape: [batch_size, 10, 2]
            return torch.stack(stacked_logits, dim=1)